# Cell marker positivity — training

A CNN that decides whether one segmented cell is positive for one IMC marker.

The input to the network is a crop centred on the cell:

* the target marker channel plus its neighbouring metal channels (`N_SIDE` on each side),
  each normalised on its own within the crop,
* the binary mask of the target cell,
* optionally the binary mask of all *other* cells in the crop (`USE_OTHER_CELLS`).

Every region comes with the same cells drawn at a range of sizes, from the bare nucleus
up to ten pixels of growth. Each sample picks one of those sizes at random, so the model
cannot learn to read positivity off how large a cell happens to be drawn.

Validation holds out whole blocks of each region, so the cells it is scored on are never
trained on, in any channel or at any size.

Nothing is written to disk. Only the marker volumes and one label map per size are held
in memory; the crops are cut out of them on the fly, every iteration.

In [ ]:
import os

# Train on the CPU even when a GPU is there, e.g. while the GPU is busy with something
# else. TensorFlow reads this when it starts, so restart the kernel after changing it.
USE_CPU = False

if USE_CPU:
    os.environ["CUDA_VISIBLE_DEVICES"] = "-1"

import numpy as np
import nibabel as nib
import tensorflow as tf
import matplotlib.pyplot as plt
from scipy import ndimage
from tensorflow.keras import layers, models

print("TensorFlow:", tf.__version__)
print("Training on:", "GPU" if tf.config.list_physical_devices("GPU") else "CPU")

# --- data -----------------------------------------------------------------
DATA_DIR = "TrainData"

# Each region is a marker stack and the same cells drawn at a range of sizes.
DATASETS = [
    ("EMMA1.nii", [
        "EMMA1_inference_cell_nuclei_mask_corrected_labled.nii",
        "EMMA1_inference_cell_nuclei_mask_corrected_labled_dilated1.nii",
        "EMMA1_inference_cell_nuclei_mask_corrected_labled_dilated2.nii",
        "EMMA1_inference_cell_nuclei_mask_corrected_labled_dilated3.nii",
        "EMMA1_inference_cell_nuclei_mask_corrected_labled_dilated4.nii",
        "EMMA1_inference_cell_nuclei_mask_corrected_labled_dilated5.nii",
        "EMMA1_inference_cell_nuclei_mask_corrected_labled_dilated6.nii",
        "EMMA1_inference_cell_nuclei_mask_corrected_labled_dilated7.nii",
        "EMMA1_inference_cell_nuclei_mask_corrected_labled_dilated8.nii",
        "EMMA1_inference_cell_nuclei_mask_corrected_labled_dilated9.nii",
        "EMMA1_inference_cell_nuclei_mask_corrected_labled_dilated10.nii",
    ]),
    ("20231201_713Lung_E_3_20231201-154853-900840_Q013_markers_channels_zstack.nii.gz", [
        "20231201_713Lung_E_3_20231201-154853-900840_Q013_cell_nuclei_mask_corrected_labled.nii",
        "20231201_713Lung_E_3_20231201-154853-900840_Q013_cell_nuclei_mask_corrected_labled_dilated1.nii",
        "20231201_713Lung_E_3_20231201-154853-900840_Q013_cell_nuclei_mask_corrected_labled_dilated2.nii",
        "20231201_713Lung_E_3_20231201-154853-900840_Q013_cell_nuclei_mask_corrected_labled_dilated3.nii",
        "20231201_713Lung_E_3_20231201-154853-900840_Q013_cell_nuclei_mask_corrected_labled_dilated4.nii",
        "20231201_713Lung_E_3_20231201-154853-900840_Q013_cell_nuclei_mask_corrected_labled_dilated5.nii",
        "20231201_713Lung_E_3_20231201-154853-900840_Q013_cell_nuclei_mask_corrected_labled_dilated6.nii",
        "20231201_713Lung_E_3_20231201-154853-900840_Q013_cell_nuclei_mask_corrected_labled_dilated7.nii",
        "20231201_713Lung_E_3_20231201-154853-900840_Q013_cell_nuclei_mask_corrected_labled_dilated8.nii",
        "20231201_713Lung_E_3_20231201-154853-900840_Q013_cell_nuclei_mask_corrected_labled_dilated9.nii",
        "20231201_713Lung_E_3_20231201-154853-900840_Q013_cell_nuclei_mask_corrected_labled_dilated10.nii",
    ]),
]

# Values used in the positivity masks.
NEGATIVE, POSITIVE, UNKNOWN = 1, 2, 3

# --- model input ----------------------------------------------------------
CROP = 64
N_SIDE = 2                 # neighbouring metal channels on each side
USE_OTHER_CELLS = True     # add the mask of the surrounding cells
PERCENTILE = 99            # every marker channel of a crop is scaled by its own percentile

N_MARKERS = 2 * N_SIDE + 1
N_CHANNELS = N_MARKERS + 1 + int(USE_OTHER_CELLS)

# --- augmentation ---------------------------------------------------------
SHIFT_CHANCE = 0.3              # how often a marker channel is shifted against the masks
MAX_SHIFT = 2                   # by up to this many pixels in y and in x

NOISE_CHANCE = 0.5              # how often counting noise is added to a crop
NOISE_COUNTS = (10, 100)        # counts a channel's percentile level stands for

SPILL_CHANCE = 0.1              # how often each marker channel spills into a neighbour
SPILL_FRACTION = (0.05, 0.5)    # size of the copy in the direct neighbour
SECOND_CHANCE = 0.5             # how often the spill reaches the second neighbour too
SECOND_FRACTION = (0.25, 0.75)  # size of that copy, relative to the first one
SPILL_NOISE = 0.3               # how rough each copy is

# --- training -------------------------------------------------------------
SEED = 42
VAL_FRAC = 0.2             # share of each region's blocks held out for validation
VAL_BLOCK = 64             # side of those blocks in pixels
BATCH_SIZE = 512
EPOCHS = 200
BASE_LR, MIN_LR, CYCLE_LENGTH = 1e-3, 1e-6, 20

print(f"Input: {CROP}x{CROP}x{N_CHANNELS}")

## Load the regions and index the labelled cells

Each region is a marker stack `(H, W, Z)` and, at every size, a positivity mask of the
same shape in which every cell is filled with 1 (negative), 2 (positive) or 3 (unknown)
for each marker channel.

The cell segmentation is the same for every marker, so the cells are labelled once per
size. Each size is then renumbered to carry the smallest mask's numbering, so one cell
can be found at any size. Every cell is uniform within a channel, so one representative
pixel is enough to read off its values for all channels at once. Cells that are unknown
in a channel are simply left out of the index.

For validation, each region is cut into `VAL_BLOCK` × `VAL_BLOCK` blocks and a share
`VAL_FRAC` of them is held out. A cell belongs to the block its nucleus centre falls in,
and a held-out cell is held out in every channel and at every size. Holding out whole
blocks rather than scattered cells keeps most of a validation cell's crop out of
training as well.

In [ ]:
class Region:
    """One imaged region held in memory, with its cells drawn at every size."""

    def __init__(self, markers, cells, centres):
        self.markers = markers   # (H, W, Z) float32, the marker stack
        self.cells = cells       # per size, (H, W) int32 cell numbers
        self.centres = centres   # per size, (n_cells, 2) int32 cell centres


def align_to_smallest(smallest, cells):
    """Give a grown mask the cell numbers the smallest mask uses.

    A connected component analysis numbers cells in the order it meets them, and that
    order changes as they grow. A grown cell covers the cell it grew from, so the cell
    of the smallest mask it shares most pixels with says which one it is.
    """
    span = int(cells.max()) + 1
    both = (smallest > 0) & (cells > 0)

    pair, count = np.unique(smallest[both].astype(np.int64) * span + cells[both],
                            return_counts=True)
    order = np.argsort(count)                # smallest overlap first, so the largest wins

    renumber = np.zeros(span, dtype=np.int32)
    renumber[(pair % span)[order]] = (pair // span)[order]

    return renumber[cells]


def load_region(image_file, mask_files):
    """Return the region at every size and, per cell and channel, its positivity."""
    markers = nib.as_closest_canonical(nib.load(f"{DATA_DIR}/{image_file}"))
    markers = markers.get_fdata().astype(np.float32)

    cells_at_size, centres_at_size, values = [], [], None

    for mask_file in mask_files:
        positivity = nib.as_closest_canonical(nib.load(f"{DATA_DIR}/{mask_file}"))
        positivity = positivity.get_fdata().astype(np.int16)

        if markers.shape != positivity.shape:
            raise ValueError(f"{mask_file}: {positivity.shape} against {markers.shape}")

        cells, n_cells = ndimage.label(positivity[:, :, 0] > 0)
        cells = cells.astype(np.int32)

        if values is None:
            # The smallest mask fixes the numbering, and the values every size shares.
            representative = np.zeros(n_cells + 1, dtype=np.int64)
            representative[cells.ravel()] = np.arange(cells.size)
            values = positivity.reshape(-1, positivity.shape[2])[representative[1:]]
        else:
            cells = align_to_smallest(cells_at_size[0], cells)

        cells_at_size.append(cells)
        centres_at_size.append(np.array(ndimage.center_of_mass(
            cells > 0, cells, np.arange(1, len(values) + 1))).astype(np.int32))

    return Region(markers, cells_at_size, centres_at_size), values


def held_out_cells(region, rng):
    """Pick the cells to validate on: every cell in a random share of the blocks."""
    height, width = region.markers.shape[:2]
    blocks_x = -(-width // VAL_BLOCK)
    n_blocks = -(-height // VAL_BLOCK) * blocks_x

    held = rng.choice(n_blocks, max(1, round(VAL_FRAC * n_blocks)), replace=False)

    centres = region.centres[0]               # the nucleus centres
    block = centres[:, 0] // VAL_BLOCK * blocks_x + centres[:, 1] // VAL_BLOCK

    return np.isin(block, held)


regions = []
index = []        # columns: region, z, cell number
labels = []       # 1.0 = positive
validation = []   # True = held out for validation

split_rng = np.random.default_rng(SEED)

for region_id, (image_file, mask_files) in enumerate(DATASETS):
    region, values = load_region(image_file, mask_files)
    regions.append(region)

    labelled = (values == NEGATIVE) | (values == POSITIVE)
    cell, z = np.nonzero(labelled)
    held = held_out_cells(region, split_rng)

    index.append(np.column_stack([np.full(cell.size, region_id), z, cell + 1]))
    labels.append(values[cell, z] == POSITIVE)
    validation.append(held[cell])

    print(f"{image_file[:40]:40s} {region.markers.shape}  "
          f"{len(region.cells):2d} sizes  {values.shape[0]:5d} cells "
          f"({held.sum()} held out)  {cell.size:6d} labelled  "
          f"{labelled.size - cell.size:5d} unknown")

index = np.concatenate(index).astype(np.int32)
labels = np.concatenate(labels).astype(np.float32)
validation = np.concatenate(validation)

memory = sum(r.markers.nbytes + sum(c.nbytes for c in r.cells) for r in regions)
print(f"\nTotal: {len(index)} samples, {labels.mean():.1%} positive, "
      f"{memory / 1e6:.0f} MB in memory")
print(f"Train: {(~validation).sum()} samples, {labels[~validation].mean():.1%} positive")
print(f"Val:   {validation.sum()} samples, {labels[validation].mean():.1%} positive")

## Cutting a sample out of a region

The crop is centred on the cell but kept inside the image, exactly as the inference
notebook does it.

Near the ends of the stack some of the neighbouring channels do not exist. Those are
left empty rather than repeating the edge channel: a copy of the target channel would
be perfectly correlated with it, which is exactly what spillover looks like, whereas
an empty channel simply says that there is no such neighbour.

Every marker channel of the crop is then normalised on its own: divided by its own
`PERCENTILE` within the crop and clipped at 1. Inference does exactly the same, so
nothing depends on the absolute counts of a region or a marker. IMC is sparse enough
that the percentile can be zero while a few pixels still carry signal; such a channel
is scaled by its maximum instead, and a channel that is empty throughout stays empty.

Every sample takes its masks from the nucleus mask or from any of the grown ones, picked
at random, which is what keeps the model from reading positivity off how large a cell
happens to be drawn. Validation samples get one size each, fixed once, so that their
scores can be compared from one epoch to the next.

Training samples are augmented on top of that, in this order:

1. **Shift.** Each marker channel, with chance `SHIFT_CHANCE`, is cut out up to
   `MAX_SHIFT` pixels off from the masks, as real channels are never perfectly aligned.
2. **Normalise**, as above.
3. **Counting noise.** With chance `NOISE_CHANCE`, every channel is redrawn as Poisson
   counts, with its percentile level standing for a random number of counts. This
   imitates a weaker acquisition.
4. **Spillover.** Each marker channel, with chance `SPILL_CHANCE`, spills into the channel
   directly before or after it, either equally likely, as a random fraction of a noisy
   copy of itself. With chance `SECOND_CHANCE` the spill also reaches the next channel
   on, with new noise and a smaller fraction. A neighbour that is not in the crop, or
   lies beyond the end of the stack, is skipped. The copies are taken from the channels
   before any spill, so spill never spills further.
5. **Normalise again.** Every channel that was changed is normalised exactly as at
   inference.
6. **Turn, flip and reverse.** The crop is rotated and flipped, and the order of the
   marker channels is reversed, which makes the model indifferent to whether the stack
   was written in ascending or descending order; the target channel sits in the middle,
   so only its neighbours swap sides.

The two mask channels are never altered, and neither are the empty channels beyond the
ends of the stack.

Every batch draws its random numbers from its own generator, seeded by the epoch and the
batch, as the batches are prepared on several threads at once.

In [ ]:
def normalise(markers):
    """Scale every marker channel of a crop by its own percentile and clip it at 1."""
    scale = np.percentile(markers, PERCENTILE, axis=(0, 1))

    # A sparse channel can have a zero percentile and still carry a little signal, which
    # its maximum then scales; a channel that is empty throughout stays empty.
    scale = np.where(scale > 0, scale, markers.max(axis=(0, 1)))
    scale[scale == 0] = 1.0

    return np.minimum(markers / scale, 1.0).astype(np.float32)


def neighbours(z, n_z):
    """The channels that make up the crop, and which of them exist in the stack."""
    wanted = np.arange(z - N_SIDE, z + N_SIDE + 1)
    return wanted, (wanted >= 0) & (wanted < n_z)


def make_sample(region_id, z, cell_id, size, rng=None):
    """The normalised crop around one cell; with `rng`, its channels are shifted at random."""
    region = regions[region_id]
    height, width, n_z = region.markers.shape
    half = CROP // 2

    centre_y, centre_x = region.centres[size][cell_id - 1]

    y = min(max(centre_y - half, 0), height - CROP)
    x = min(max(centre_x - half, 0), width - CROP)

    cells = region.cells[size][y:y + CROP, x:x + CROP]

    # Channels beyond either end of the stack do not exist and stay empty.
    wanted, exists = neighbours(z, n_z)

    markers = np.zeros((CROP, CROP, N_MARKERS), dtype=np.float32)
    for i in np.flatnonzero(exists):
        dy, dx = 0, 0
        if rng is not None and rng.random() < SHIFT_CHANCE:
            dy, dx = rng.integers(-MAX_SHIFT, MAX_SHIFT + 1, size=2)

        yy = min(max(y + dy, 0), height - CROP)
        xx = min(max(x + dx, 0), width - CROP)
        markers[..., i] = region.markers[yy:yy + CROP, xx:xx + CROP, wanted[i]]

    planes = [normalise(markers), (cells == cell_id)[..., None]]
    if USE_OTHER_CELLS:
        planes.append(((cells > 0) & (cells != cell_id))[..., None])

    # concatenating the float32 markers with the boolean masks already gives float32
    return np.concatenate(planes, axis=-1, dtype=np.float32)


def add_counting_noise(markers, exists, rng):
    """Redraw every channel as Poisson counts, as if fewer ions had been counted."""
    for i in np.flatnonzero(exists):
        counts = rng.uniform(*NOISE_COUNTS)
        markers[..., i] = rng.poisson(markers[..., i] * counts) / counts

    return markers


def add_spillover(markers, exists, rng):
    """Let each channel, now and then, spill into the channels beside it.

    The spill goes to the channel directly before or after, and sometimes on to the
    one after that, more weakly. Showing the network what this looks like lets it tell
    a faint echo of a strong neighbour apart from real expression.

    Returns the channels and which of them received spill.
    """
    source = markers.copy()          # spill comes from the channels as they were measured
    changed = np.zeros(N_MARKERS, dtype=bool)

    for channel in np.flatnonzero(exists):
        if rng.random() >= SPILL_CHANCE:
            continue

        step = rng.choice((-1, 1))
        fraction = rng.uniform(*SPILL_FRACTION)
        targets = [(channel + step, fraction)]
        if rng.random() < SECOND_CHANCE:
            targets.append((channel + 2 * step, fraction * rng.uniform(*SECOND_FRACTION)))

        for target, fraction in targets:
            if not (0 <= target < N_MARKERS and exists[target]):
                continue                    # no such neighbour, so nothing to spill into

            noise = 1.0 + SPILL_NOISE * rng.standard_normal((CROP, CROP), dtype=np.float32)
            markers[..., target] += fraction * source[..., channel] * np.maximum(noise, 0.0)
            changed[target] = True

    return markers, changed


def augment(sample, exists, rng):
    """Add counting noise and spillover, renormalise, then turn, flip and reverse."""
    markers = sample[..., :N_MARKERS].copy()
    changed = np.zeros(N_MARKERS, dtype=bool)

    if rng.random() < NOISE_CHANCE:
        markers = add_counting_noise(markers, exists, rng)
        changed |= exists

    markers, spilled = add_spillover(markers, exists, rng)
    changed |= spilled

    # Only the channels that changed are normalised again, exactly as at inference.
    if changed.any():
        markers[..., changed] = normalise(markers[..., changed])

    sample = np.concatenate([markers, sample[..., N_MARKERS:]], axis=-1)

    sample = np.rot90(sample, rng.integers(4), axes=(0, 1))
    if rng.random() < 0.5:
        sample = sample[:, ::-1]
    if rng.random() < 0.5:
        sample = np.concatenate(
            [sample[..., N_MARKERS - 1::-1], sample[..., N_MARKERS:]], axis=-1)

    return np.ascontiguousarray(sample, dtype=np.float32)


class CellSequence(tf.keras.utils.Sequence):
    """Batches of samples; `sizes` fixes the mask size of every row, otherwise random."""

    def __init__(self, rows, augment_samples, sizes=None):
        self.rows = rows
        self.augment_samples = augment_samples
        self.sizes = sizes
        self.epoch = -1
        self.order_rng = np.random.default_rng(SEED)
        self.on_epoch_end()

    def __len__(self):
        return int(np.ceil(len(self.rows) / BATCH_SIZE))

    def on_epoch_end(self):
        self.epoch += 1
        if self.augment_samples:
            self.order = self.order_rng.permutation(len(self.rows))
        else:
            self.order = np.arange(len(self.rows))

    def __getitem__(self, batch):
        # one generator per batch, since the batches are prepared on several threads
        rng = np.random.default_rng((SEED, self.epoch, batch))

        positions = self.order[batch * BATCH_SIZE:(batch + 1) * BATCH_SIZE]
        rows = self.rows[positions]
        x = np.empty((len(rows), CROP, CROP, N_CHANNELS), dtype=np.float32)

        for i, (position, row) in enumerate(zip(positions, rows)):
            region_id, z, cell_id = index[row]
            region = regions[region_id]

            if self.sizes is not None:
                size = self.sizes[position]
            else:
                size = rng.integers(len(region.cells))   # the nucleus or any grown mask

            if self.augment_samples:
                sample = make_sample(region_id, z, cell_id, size, rng)
                exists = neighbours(z, region.markers.shape[2])[1]
                x[i] = augment(sample, exists, rng)
            else:
                x[i] = make_sample(region_id, z, cell_id, size)

        return x, labels[rows]


train_rows = np.flatnonzero(~validation)
val_rows = np.flatnonzero(validation)

# every validation sample keeps one mask size, picked once
val_sizes = np.random.default_rng(SEED).integers(
    [len(regions[region_id].cells) for region_id in index[val_rows, 0]])

train_gen = CellSequence(train_rows, augment_samples=True)
val_gen = CellSequence(val_rows, augment_samples=False, sizes=val_sizes)

print(f"Train: {len(train_gen.rows)} samples, val: {len(val_gen.rows)} samples")

## Check a sample

Every input channel of one random cell, so it is easy to see that the marker
channels, the cell mask and the surrounding cells line up. Below it, the same cell at
every size, and a few augmented versions of it as the training sees them.

In [ ]:
row = np.random.default_rng(0).integers(len(index))
region_id, z, cell_id = index[row]
sample = make_sample(region_id, z, cell_id, 0)

names = [f"marker z{z:+d}" for z in range(-N_SIDE, N_SIDE + 1)] + ["this cell"]
if USE_OTHER_CELLS:
    names.append("other cells")

fig, axes = plt.subplots(1, N_CHANNELS, figsize=(3 * N_CHANNELS, 3.4))
for ax, name, plane in zip(axes, names, np.moveaxis(sample, -1, 0)):
    ax.imshow(plane, cmap="gray", vmin=0, vmax=1)
    ax.set_title(name, fontsize=10)
    ax.axis("off")

fig.suptitle(f"region {region_id}, channel {z}, cell {cell_id} — "
             f"{'POSITIVE' if labels[row] else 'NEGATIVE'}")
plt.tight_layout()
plt.show()

# the same cell at every size, which is what a sample picks from at random
sizes = range(len(regions[region_id].cells))
fig, axes = plt.subplots(1, len(sizes), figsize=(1.7 * len(sizes), 2.2))
for size, ax in zip(sizes, axes):
    ax.imshow(make_sample(region_id, z, cell_id, size)[..., N_MARKERS], cmap="gray")
    ax.set_title(f"+{size}", fontsize=9)
    ax.axis("off")
fig.suptitle("the same cell, grown one pixel at a time")
plt.tight_layout()
plt.show()

# a few augmented versions, one per row, marker channels only
exists = neighbours(z, regions[region_id].markers.shape[2])[1]
rng = np.random.default_rng(1)
fig, axes = plt.subplots(4, N_MARKERS, figsize=(2.2 * N_MARKERS, 2.2 * 4))
for row_axes in axes:
    augmented = augment(make_sample(region_id, z, cell_id, 0, rng), exists, rng)
    for ax, name, plane in zip(row_axes, names, np.moveaxis(augmented, -1, 0)):
        ax.imshow(plane, cmap="gray", vmin=0, vmax=1)
        ax.set_title(name, fontsize=8)
        ax.axis("off")
fig.suptitle("augmented versions (turned, flipped and possibly channel-reversed)")
plt.tight_layout()
plt.show()

## Model

In [ ]:
def conv_bn_relu(x, filters, kernel_size=3, strides=1):
    x = layers.Conv2D(filters, kernel_size, strides=strides,
                      padding="same", use_bias=False)(x)
    x = layers.BatchNormalization()(x)
    return layers.Activation("relu")(x)


def make_marker_cnn(input_shape):
    inputs = layers.Input(shape=input_shape)

    c = 32
    x = conv_bn_relu(inputs, c)
    x = conv_bn_relu(x, c)
    x = conv_bn_relu(x, c)
    x = conv_bn_relu(x, c)

    x = conv_bn_relu(x, 2 * c, strides=2)
    x = conv_bn_relu(x, 4 * c, strides=2)
    x = conv_bn_relu(x, 8 * c, strides=2)

    x = layers.Conv2D(1, 1)(x)
    x = layers.GlobalAveragePooling2D()(x)
    outputs = layers.Activation("sigmoid")(x)

    model = models.Model(inputs, outputs)
    model.compile(
        optimizer=tf.keras.optimizers.Adam(learning_rate=BASE_LR),
        loss="binary_crossentropy",
        metrics=["accuracy", tf.keras.metrics.AUC(name="auc")],
    )
    return model


model = make_marker_cnn((CROP, CROP, N_CHANNELS))
model.summary()

## Train

In [ ]:
import os

os.makedirs("checkpoints", exist_ok=True)


def cosine_annealing_with_restarts(epoch, lr):
    t = (epoch % CYCLE_LENGTH) / CYCLE_LENGTH
    return MIN_LR + (BASE_LR - MIN_LR) * 0.5 * (1 + np.cos(np.pi * t))


callbacks = [
    tf.keras.callbacks.LearningRateScheduler(cosine_annealing_with_restarts),
    tf.keras.callbacks.CSVLogger("training_log.csv"),     # a fresh log for every run
    tf.keras.callbacks.ModelCheckpoint("checkpoints/marker_cnn_epoch_{epoch:03d}.h5"),
]

history = model.fit(
    train_gen,
    validation_data=val_gen,
    epochs=EPOCHS,
    callbacks=callbacks,
    workers=8,
    use_multiprocessing=False,
)

model.save("marker_cnn_final.h5")
print("Training complete.")

## Training curves

In [ ]:
for key in ["loss", "accuracy", "auc"]:
    plt.figure()
    plt.plot(history.history[key], label=f"train {key}")
    plt.plot(history.history[f"val_{key}"], label=f"val {key}")
    plt.xlabel("epoch")
    plt.ylabel(key)
    plt.legend()
    plt.grid(True)
    plt.show()

## Architecture figure (optional)

Needs `pip install visualkeras pillow`.

In [ ]:
import visualkeras
from PIL import Image
from IPython.display import Image as IPyImage


def text_callable(layer_index, layer):
    shape = [s for s in layer.output.shape if s is not None]
    text = "x".join(str(s) for s in shape[:-1]) + f"\n{shape[-1]}" if shape else "?"
    return f"{text}\n{layer.name}", bool(layer_index % 2)


image = visualkeras.layered_view(
    model, legend=True, show_dimension=True, spacing=15,
    draw_volume=True, scale_xy=0.8, text_callable=text_callable,
)
image = image.resize((int(image.width * 0.6), image.height), Image.BILINEAR)
image.save("marker_cnn_3d.png")

IPyImage("marker_cnn_3d.png")